# optimizer-loop-on-tensor — worked example 3: None-grad guard handles frozen and unfrozen params in the same step

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `optimizer-loop-on-tensor`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

The `if p.grad is not None` guard inside `.step()` is essential when some parameters are frozen (`requires_grad=False`) or simply did not participate in the most recent forward pass. Frozen parameters never accumulate a gradient, so `p.grad` stays `None`. Without the guard, attempting `p -= lr * None` raises a `TypeError`.

## Worked solution

**Step 1 — Set up mixed parameters.**
We create two tensors: `active` (requires_grad=True, grad will be populated) and `frozen` (requires_grad=False, grad stays None).

**Step 2 — Populate grads manually.**
We set `active.grad = t.ones(3)`. We deliberately leave `frozen.grad = None`.

**Step 3 — Run the step.**
The optimizer iterates both. For `active`, the guard passes and the update runs. For `frozen`, the guard blocks execution and the tensor is unchanged.

**Step 4 — Verify behavior.**
After the step: `active` has moved (its value decreased by `lr * 1.0`). `frozen` is unchanged.

In [ ]:
import torch as t

class GuardedSGD:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr

    @t.inference_mode()
    def step(self):
        for p in self.params:
            if p.grad is None:
                continue   # skip frozen or non-participating params
            p -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = None

# --- exercise it ---
t.manual_seed(1)
active = t.tensor([1.0, 2.0, 3.0], requires_grad=True)
frozen = t.tensor([5.0, 6.0, 7.0])  # requires_grad=False by default

# Pass BOTH to the optimizer
opt = GuardedSGD([active, frozen], lr=0.1)

# Manually set grad only on active
active.grad = t.ones(3)
# frozen.grad remains None

active_before = active.data.clone()
frozen_before = frozen.clone()

opt.step()

print(f'active before: {active_before.tolist()}')
print(f'active after : {active.data.tolist()}')
print(f'frozen before: {frozen_before.tolist()}')
print(f'frozen after : {frozen.tolist()}')

# active should have moved by lr * 1.0 = 0.1
assert t.allclose(active.data, active_before - 0.1 * t.ones(3))
assert t.allclose(frozen, frozen_before), 'Frozen tensor should not change'